# WUMI fire perimeters

Downloads real, agency-mapped fire perimeters for one state from the
[Western US MTBS-Interagency Database of Large Wildfires (WUMI)](https://datadryad.org/dataset/doi:10.5061/dryad.63xsj3vd4),
which merges MTBS, CalFire, USGS, WFIGS, and IAFPH into one record per fire from 1984 to
the present. Feeds [`fire_overlap.ipynb`](fire_overlap.ipynb) -- the same relationship
[`burn_prob.ipynb`](burn_prob.ipynb) has to [`risk.ipynb`](risk.ipynb).

**This notebook was written by generative AI.**

## Why this isn't just another view in `views.sql`

Dryad's _website_ downloads sit behind [Anubis](https://github.com/TecharoHQ/anubis), a
bot-detection proof-of-work challenge that only a real browser can solve, so a scripted
request to the page a human would click 403s. (`views.sql` hits the same kind of wall for
the census state boundaries -- a WAF there instead of Anubis -- with the same fix: a
local copy.)

Dryad's [REST API](https://datadryad.org/api) sits in front of the same files and has no
such wall -- but downloading through it needs a bearer token, which needs a Dryad API
account: log in at [datadryad.org](https://datadryad.org) with an ORCID, then create one
under [My account](https://datadryad.org/account), which gives a client ID and secret for
the OAuth `client_credentials` grant. Put them in `.env` as `DRYAD_CLIENT_ID` and
`DRYAD_SECRET` (see `.env.sample`) before running this notebook.

The other complication is structural. `fire_maps.zip` (822 MB) is a zip of one zip per
year, each of which contains one folder per fire, holding a shapefile from every source
that mapped it (`burnarea_mtbs.shp`, `burnarea_usgs.shp`, ...) plus a
`burnarea_circle.shp` fallback -- the fire's perimeter _assuming it was a circle centered
on the ignition point_, used when no agency ever mapped a real boundary. That fallback is
a guess, not an observation, so it is excluded below: a fire only makes it into the cache
if at least one real source mapped it.

Everything below only ever fetches the bytes it needs: the outer zip's directory to find
each year, each year's directory to find fires whose ignition point falls in the state's
(padded) bounding box -- the fire ID itself encodes that point, see `fireid_latlon` --
and then only the shapefile members for fires that pass the filter. For Wyoming that is a
few dozen megabytes out of 822, most of it spent listing directories rather than reading
shapes.


## Configuration


In [1]:
STATE = "WY"  # must match what fire_overlap.ipynb is configured to read

DRYAD_DOI = "10.5061/dryad.63xsj3vd4"
DRYAD_FILENAME = "fire_maps.zip"

# a fire's ignition point has to fall within this many degrees of the state's bounding
# box; its perimeter can still extend further, the same reasoning as BBOX_PAD_DEG in
# risk.ipynb
BBOX_PAD_DEG = 0.5

# preference order when more than one source mapped the same fire
SOURCE_PRIORITY = ["mtbs", "usgs", "wfigs", "iafph", "calfire"]

MAX_WORKERS = 16

OUT_PATH = f"data/wumi_perimeters/{STATE}.parquet"

### The state's bounding box


In [2]:
import geopandas as gpd

state_bounds = gpd.read_file("zip://data/cb_2018_us_state_20m.zip")
state_row = state_bounds[state_bounds["STUSPS"] == STATE]
min_lon, min_lat, max_lon, max_lat = state_row.total_bounds
BBOX = (
    min_lon - BBOX_PAD_DEG,
    min_lat - BBOX_PAD_DEG,
    max_lon + BBOX_PAD_DEG,
    max_lat + BBOX_PAD_DEG,
)
BBOX

(np.float64(-111.556888),
 np.float64(40.496345999999996),
 np.float64(-103.552287),
 np.float64(45.505904))

## Getting a download URL from the Dryad API

Three calls: trade the API credentials for a bearer token, look up `DRYAD_FILENAME`'s
current file ID for `DRYAD_DOI` (rather than hardcoding one, which would break the next
time WUMI publishes a new version), then ask that file's `download` link for its bytes.
That last call redirects to a presigned S3 URL good for about 24 hours -- resolving it
once up front means `RangeReader` below never has to know a Dryad token exists; it just
sees a URL that answers `Range` requests.


In [3]:
import os
from urllib.parse import quote

import httpx
from dotenv import load_dotenv

load_dotenv()

DRYAD_API = "https://datadryad.org/api/v2"


def get_dryad_token(client_id: str, client_secret: str) -> str:
    response = httpx.post(
        "https://datadryad.org/oauth/token",
        data={
            "client_id": client_id,
            "client_secret": client_secret,
            "grant_type": "client_credentials",
        },
        timeout=30,
    )
    response.raise_for_status()
    return response.json()["access_token"]


token = get_dryad_token(os.environ["DRYAD_CLIENT_ID"], os.environ["DRYAD_SECRET"])

In [4]:
def resolve_download_url(token: str, doi: str, filename: str) -> str:
    """The current presigned URL for `filename` in the latest version of `doi`."""
    headers = {"Authorization": f"Bearer {token}"}
    dataset = httpx.get(
        f"{DRYAD_API}/datasets/{quote(doi, safe='')}", headers=headers, timeout=30
    ).json()
    version_href = dataset["_links"]["stash:version"]["href"]

    files = httpx.get(
        f"https://datadryad.org{version_href}/files", headers=headers, timeout=30
    ).json()
    matches = [f for f in files["_embedded"]["stash:files"] if f["path"] == filename]
    if not matches:
        available = [f["path"] for f in files["_embedded"]["stash:files"]]
        raise ValueError(f"{filename!r} not found in {doi} -- have {available}")
    download_href = matches[0]["_links"]["stash:download"]["href"]

    # a HEAD-sized Range request is enough to follow the redirect to the presigned URL
    # without pulling any of the file itself
    response = httpx.get(
        f"https://datadryad.org{download_href}",
        headers={**headers, "Range": "bytes=0-0"},
        follow_redirects=True,
        timeout=30,
    )
    response.raise_for_status()
    return str(response.url)


FIRE_MAPS_SOURCE = resolve_download_url(token, DRYAD_DOI, DRYAD_FILENAME)

## Finding fires in the box, without downloading the archive

A fire ID is `<ignition date>_<north latitude>_<west longitude>`, with the coordinates
multiplied by 10,000 and rounded (see the WUMI README) -- so filtering candidates is
string parsing on folder names, not a spatial operation.

Reading those folder names still means listing each year's directory. `zipfile` can do
that against a plain file object, whether that object is a local file or something that
fetches on demand -- so `RangeReader` below fakes a seekable file over an HTTP resource
using nothing but `Range` requests. `zipfile.ZipFile.open("fire_maps/1984.zip")` handles
finding that member's bytes inside the outer zip on its own; nesting it in a second
`zipfile.ZipFile` reads _its_ directory the same way, without either zip being fully
downloaded.


In [5]:
import io


class RangeReader(io.RawIOBase):
    """A read-only seekable file over an HTTP resource, fetched lazily via Range
    requests -- lets `zipfile` read a remote archive's directory without downloading it."""

    def __init__(self, client: httpx.Client, url: str):
        self.client, self.url, self.pos = client, url, 0
        response = client.get(url, headers={"Range": "bytes=0-0"})
        self.size = int(response.headers["content-range"].split("/")[-1])

    def readable(self):
        return True

    def seekable(self):
        return True

    def seek(self, offset, whence=0):
        self.pos = {0: offset, 1: self.pos + offset, 2: self.size + offset}[whence]
        return self.pos

    def tell(self):
        return self.pos

    def readinto(self, b):
        end = min(self.pos + len(b), self.size) - 1
        if end < self.pos:
            return 0
        data = self.client.get(
            self.url, headers={"Range": f"bytes={self.pos}-{end}"}
        ).content
        b[: len(data)] = data
        self.pos += len(data)
        return len(data)


def open_zip_source(source: str):
    if source.startswith(("http://", "https://")):
        return RangeReader(httpx.Client(timeout=60), source)
    return open(source, "rb")

In [6]:
import re

FIREID_RE = re.compile(r"^(\d{8})_(\d+)_(\d+)$")


def fireid_latlon(fireid: str) -> tuple[float, float]:
    """A fire's ignition point, decoded from its ID."""
    _, latx, lonx = FIREID_RE.match(fireid).groups()
    return int(latx) / 10000, -int(lonx) / 10000


ENTRY_RE = re.compile(r"^(\d{8}_\d+_\d+)/burnarea_(\w+)\.shp$")


def discover_candidates(source: str, bbox: tuple[float, float, float, float]) -> dict:
    """Every fire ID whose ignition point falls in `bbox`, and which sources mapped it."""
    import zipfile

    outer = zipfile.ZipFile(open_zip_source(source))
    years = sorted(
        int(m.group(1))
        for name in outer.namelist()
        if (m := re.match(r"fire_maps/(\d{4})\.zip$", name))
    )

    candidates: dict[str, dict] = {}
    for year in years:
        year_zip = zipfile.ZipFile(outer.open(f"fire_maps/{year}.zip"))
        for name in year_zip.namelist():
            m = ENTRY_RE.match(name)
            if not m:
                continue
            fireid, source_name = m.groups()
            lat, lon = fireid_latlon(fireid)
            if bbox[0] <= lon <= bbox[2] and bbox[1] <= lat <= bbox[3]:
                candidates.setdefault(fireid, {"year": year, "sources": set()})[
                    "sources"
                ].add(source_name)
    return candidates

In [7]:
from time import monotonic

t0 = monotonic()
candidates = discover_candidates(FIRE_MAPS_SOURCE, BBOX)
print(f"{len(candidates)} candidate fires in the box ({monotonic() - t0:.0f}s)")

2419 candidate fires in the box (27s)


## Circular vs. real perimeters

Each candidate's folder may hold several sources' shapefiles for the same fire, plus the
circular fallback. `best_source` picks the highest-priority _real_ one, or `None` if
nobody ever mapped this fire -- those are the ones this notebook drops.


In [8]:
def best_source(sources: set[str], priority: list[str]) -> str | None:
    real = [s for s in priority if s in sources]
    return real[0] if real else None


to_fetch = {
    fireid: (info["year"], source)
    for fireid, info in candidates.items()
    if (source := best_source(info["sources"], SOURCE_PRIORITY)) is not None
}
print(f"{len(to_fetch)} of {len(candidates)} candidates have a real perimeter")

1328 of 2419 candidates have a real perimeter


## Fetching the matched shapefiles

GDAL's `/vsizip/` and `/vsicurl/` handlers can be chained, so a shapefile two zips deep on
a remote server is one path string -- no manual byte-range math needed here, unlike the
directory listing above (which GDAL has no API for).

The shapefiles' attribute tables carry several `object_ID_*` columns that get truncated
to the same 10 characters by the `.dbf` format itself, so reading every column raises a
duplicate-name error. `columns=` sidesteps it by asking for only the ones that stay
unique.


In [9]:
def fire_shp_path(zip_source: str, year: int, fireid: str, source: str) -> str:
    """GDAL path into one fire's shapefile, chaining vsizip twice: fire_maps.zip contains
    one zip per year, which contains one folder per fire."""
    base = f"/vsicurl/{zip_source}" if zip_source.startswith("http") else zip_source
    annual_zip = f"/vsizip/{{{base}}}/fire_maps/{year}.zip"
    return f"/vsizip/{{{annual_zip}}}/{fireid}/burnarea_{source}.shp"


ATTR_COLUMNS = [
    "fireid",
    "dataset",
    "agency",
    "name",
    "date",
    "poly_area_h",
    "burn_area_h",
    "cause_human",
    "cause_speci",
]

In [10]:
from concurrent.futures import ThreadPoolExecutor, as_completed

import geopandas as gpd
from tqdm.auto import tqdm


def fetch_one(fireid: str, year: int, source: str) -> gpd.GeoDataFrame:
    gdf = gpd.read_file(
        fire_shp_path(FIRE_MAPS_SOURCE, year, fireid, source), columns=ATTR_COLUMNS
    )
    gdf["fireid"] = fireid
    gdf["source"] = source
    return gdf


t0 = monotonic()
frames, errors = [], []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futures = {
        pool.submit(fetch_one, fireid, year, source): fireid
        for fireid, (year, source) in to_fetch.items()
    }
    progress = tqdm(
        as_completed(futures),
        total=len(futures),
        desc="fetching perimeters",
        mininterval=2,  # a saved notebook shouldn't carry one line per future
    )
    for future in progress:
        try:
            frames.append(future.result())
        except Exception as e:  # noqa: BLE001 -- one fire's fetch failing shouldn't abort the rest
            errors.append((futures[future], str(e)))

print(
    f"{len(frames)} perimeters fetched, {len(errors)} failed ({monotonic() - t0:.0f}s)"
)
if errors:
    print(errors[:5])

fetching perimeters:   0%|          | 0/1328 [00:00<?, ?it/s]

1328 perimeters fetched, 0 failed (106s)


## Assembling and caching


In [11]:
import pandas as pd

# no .prj ships with these shapefiles -- the README states the projection directly
fires = gpd.GeoDataFrame(pd.concat(frames, ignore_index=True)).set_crs(
    "EPSG:5070", allow_override=True
)
fires["source"].value_counts()

source
mtbs     697
usgs     461
wfigs    167
iafph      3
Name: count, dtype: int64

In [12]:
from pathlib import Path

Path(OUT_PATH).parent.mkdir(parents=True, exist_ok=True)
fires.to_parquet(OUT_PATH)
print(f"wrote {len(fires)} perimeters to {OUT_PATH}")

wrote 1328 perimeters to data/wumi_perimeters/WY.parquet


## Smoke test

Pure logic, no network -- checks the parsing and path-building this notebook depends on
without needing a copy of `fire_maps.zip` on hand.


In [13]:
assert fireid_latlon("19840126_337090_1176180") == (33.709, -117.618)

assert best_source({"circle"}, SOURCE_PRIORITY) is None
assert best_source({"circle", "usgs"}, SOURCE_PRIORITY) == "usgs"
assert best_source({"circle", "usgs", "mtbs"}, SOURCE_PRIORITY) == "mtbs"

assert (
    fire_shp_path("data/fire_maps.zip", 1984, "X", "mtbs")
    == "/vsizip/{/vsizip/{data/fire_maps.zip}/fire_maps/1984.zip}/X/burnarea_mtbs.shp"
)
assert (
    fire_shp_path("https://example.com/f.zip", 1984, "X", "mtbs")
    == "/vsizip/{/vsizip/{/vsicurl/https://example.com/f.zip}/fire_maps/1984.zip}/X/burnarea_mtbs.shp"
)

print("smoke test passed")

smoke test passed


## What this does not tell you

- **Circular-fallback fires are dropped entirely.** They are WUMI's guess at a shape, not
  an observation, and including them would quietly overstate how much is actually known.
  That drops smaller and older fires disproportionately, since they were less likely to
  get a real perimeter mapped in the first place.
- **The filter is on the ignition point, not the perimeter.** A fire that ignited just
  outside the padded bounding box but burned into the state would be missed; one that
  ignited just inside but burned out would be kept complete. `BBOX_PAD_DEG` trades one
  kind of miss for the other rather than eliminating it.
- **Mixed methodology.** MTBS perimeters come from satellite-derived burn severity;
  CalFire, USGS, WFIGS, and IAFPH perimeters are digitized boundaries from incident
  reports. They are not drawn to the same standard, which matters most right at fire
  edges -- exactly where overlap gets measured in `fire_overlap.ipynb`.
- **Needs a Dryad API account.** Without `DRYAD_CLIENT_ID`/`DRYAD_SECRET` in `.env`, the
  token request fails before anything else runs -- this isn't optional the way most of
  this project's data sources are.
- **`EPSG:5070` is CONUS-specific**, same caveat as `risk.ipynb`.
